# LangChain Tool Base Reference

Developer-facing statements defined in `langchain_core.tools.base`.

# `FILTERED_ARGS`

Contains function parameters excluded from automatically generated tool-input schemas.

```python
FILTERED_ARGS: tuple[str, str] = ("run_manager", "callbacks") # Parameters excluded from generated tool schemas
```

# `TOOL_MESSAGE_BLOCK_TYPES`

Contains supported structured content-block types accepted in tool-message output.

```python
TOOL_MESSAGE_BLOCK_TYPES: tuple[str, ...] # Supported structured ToolMessage content-block types
```

Supported values include:

```text
text
image_url
image
json
search_result
custom_tool_call_output
document
file
```

---

# `SchemaAnnotationError: TypeError`

Raised when a `BaseTool` subclass declares `args_schema` using an invalid annotation.

A common invalid declaration is:

```python
args_schema: BaseModel # Incorrect because the schema must be a model class
```

The expected form is:

```python
args_schema: type[BaseModel] # Correct annotation for a Pydantic schema class
```

---

# `create_schema_from_function`

Creates a Pydantic model from a Python function signature.

```python
create_schema_from_function(
    model_name: str, # Name assigned to the generated Pydantic model
    func: Callable[..., Any], # Function whose parameters define the schema
    *,
    filter_args: Sequence[str] | None = None, # Parameters excluded from the generated schema
    parse_docstring: bool = False, # Whether Google-style docstrings provide field descriptions
    error_on_invalid_docstring: bool = False, # Whether invalid parsed docstrings raise ValueError
    include_injected: bool = True, # Whether runtime-injected arguments remain in the validation schema
) -> TypeBaseModel # Return the generated Pydantic model class
```

## Behaviour

- Uses function annotations to determine field types.
- Uses the function docstring as the model description.
- Can parse Google-style argument descriptions.
- Excludes `run_manager` and `callbacks` by default.
- Excludes `self` or `cls` for methods.
- Supports Pydantic v1 and v2 annotations.
- Raises `NotImplementedError` when one function mixes Pydantic v1 and v2 model annotations.
- Removes synthetic Pydantic fields generated for unused `*args` and `**kwargs`.

---

# `ToolException: Exception`

Represents an expected tool-execution failure that an agent may handle without stopping its entire execution.

When `BaseTool.handle_tool_error` is configured, the exception is converted into tool output instead of being raised.

---

# `ArgsSchema`

Represents either a Pydantic model class or a JSON-schema dictionary used to validate tool input.

```python
ArgsSchema = TypeBaseModel | dict[str, Any] # Supported tool argument-schema representations
```

---

# `MessageContentBlock`

Represents one plain-text or structured tool-message content block.

```python
MessageContentBlock = str | dict[str, Any] # One tool-message content block
```

A dictionary is considered valid message content at runtime only when its `type` belongs to `TOOL_MESSAGE_BLOCK_TYPES`.

---

# `ToolExceptionHandlerOutput`

Represents content returned by a callable `handle_tool_error` handler.

```python
ToolExceptionHandlerOutput = str | Sequence[MessageContentBlock] # Handled tool-error output content
```

---

# `BaseTool: RunnableSerializable[str | dict[str, Any] | ToolCall, Any]`

`BaseTool` is the abstract base class for LangChain tools.

A concrete subclass defines tool metadata and implements `_run()` for synchronous execution. It may also override `_arun()` for a native asynchronous implementation.

## Fields

```python
name: str # Unique tool name describing its purpose
description: str # Description telling a model when and how to use the tool
args_schema: ArgsSchema | None = None # Pydantic model class or JSON schema used to validate input
return_direct: bool = False # Whether an AgentExecutor stops after receiving this tool result
verbose: bool = False # Whether tool execution logs progress
callbacks: Callbacks = None # Callback handlers used during tool execution
tags: list[str] | None = None # Tags attached to each tool execution
metadata: dict[str, Any] | None = None # Metadata attached to each tool execution
handle_tool_error: bool | str | Callable[[ToolException], ToolExceptionHandlerOutput] | None = False # ToolException handling strategy
handle_validation_error: bool | str | Callable[[ValidationError | ValidationErrorV1], str] | None = False # Input-validation error handling strategy
response_format: Literal["content", "content_and_artifact"] = "content" # Expected tool response format
extras: dict[str, Any] | None = None # Optional provider-specific tool fields
```

## Constructor

```python
BaseTool(
    *,
    name: str, # Unique tool name
    description: str, # Tool purpose and usage description
    args_schema: ArgsSchema | None = None, # Input-validation schema
    return_direct: bool = False, # Whether agent execution stops after this tool
    verbose: bool = False, # Whether execution progress is logged
    callbacks: Callbacks = None, # Tool callback handlers
    tags: list[str] | None = None, # Tool execution tags
    metadata: dict[str, Any] | None = None, # Tool execution metadata
    handle_tool_error: bool | str | Callable[[ToolException], ToolExceptionHandlerOutput] | None = False, # ToolException handling strategy
    handle_validation_error: bool | str | Callable[[ValidationError | ValidationErrorV1], str] | None = False, # Validation-error handling strategy
    response_format: Literal["content", "content_and_artifact"] = "content", # Tool output format
    extras: dict[str, Any] | None = None, # Provider-specific tool configuration
    **kwargs: Any, # Additional serializable model fields
) -> None # Initialize the tool
```

The constructor raises `TypeError` when `args_schema` is neither a Pydantic model class nor a JSON-schema dictionary.

## Properties

### `is_single_input`

Returns `True` when the tool schema contains exactly one user-provided input field.

### `args`

Returns the JSON-schema properties describing the tool's user-provided arguments.

### `tool_call_schema`

Returns the schema exposed to language models for tool calling.

Runtime-injected arguments are excluded from this schema.

For a JSON-schema dictionary, the tool description is added when available.

## Abstract and Customizable Execution Hooks

### `_run`

Required synchronous implementation supplied by a concrete tool subclass.

A subclass may accept `run_manager` to receive synchronous tool callbacks.

```python
_run(
    self, # Current tool instance
    *args: Any, # Validated positional tool arguments
    **kwargs: Any, # Validated keyword tool arguments
) -> Any # Return the tool result
```

### `_arun`

Optional asynchronous implementation.

The default implementation runs `_run()` in an executor.

A subclass may accept an asynchronous `run_manager`.

```python
async _arun(
    self, # Current tool instance
    *args: Any, # Validated positional tool arguments
    **kwargs: Any, # Validated keyword tool arguments
) -> Any # Return the asynchronous tool result
```

## Overridden Runnable Methods

### `get_input_schema`

Returns `args_schema` when a Pydantic schema class is configured.

Otherwise, it generates a schema from `_run()`.

### `invoke`

Prepares Runnable configuration and synchronously executes the tool through `run()`.

It accepts a string, argument dictionary, or complete `ToolCall`.

### `ainvoke`

Prepares Runnable configuration and asynchronously executes the tool through `arun()`.

It accepts a string, argument dictionary, or complete `ToolCall`.

## Methods

### `run`

Validates and synchronously executes the tool.

```python
run(
    self, # Current tool instance
    tool_input: str | dict[str, Any], # Raw string or dictionary tool input
    verbose: bool | None = None, # Optional per-call verbosity override
    start_color: str | None = "green", # Callback colour used when execution starts
    color: str | None = "green", # Callback colour used when execution ends
    callbacks: Callbacks = None, # Per-call callbacks
    *,
    tags: list[str] | None = None, # Per-call tags
    metadata: dict[str, Any] | None = None, # Per-call metadata
    run_name: str | None = None, # Optional trace run name
    run_id: uuid.UUID | None = None, # Optional trace run identifier
    config: RunnableConfig | None = None, # Runnable runtime configuration
    tool_call_id: str | None = None, # Tool-call identifier used for ToolMessage output
    **kwargs: Any, # Additional callback arguments
) -> Any # Return the formatted tool output
```

### `arun`

Validates and asynchronously executes the tool.

```python
async arun(
    self, # Current tool instance
    tool_input: str | dict[str, Any], # Raw string or dictionary tool input
    verbose: bool | None = None, # Optional per-call verbosity override
    start_color: str | None = "green", # Callback colour used when execution starts
    color: str | None = "green", # Callback colour used when execution ends
    callbacks: Callbacks = None, # Per-call callbacks
    *,
    tags: list[str] | None = None, # Per-call tags
    metadata: dict[str, Any] | None = None, # Per-call metadata
    run_name: str | None = None, # Optional trace run name
    run_id: uuid.UUID | None = None, # Optional trace run identifier
    config: RunnableConfig | None = None, # Runnable runtime configuration
    tool_call_id: str | None = None, # Tool-call identifier used for ToolMessage output
    **kwargs: Any, # Additional callback arguments
) -> Any # Return the formatted asynchronous tool output
```

## Input Behaviour

- String input is passed as the first positional tool argument.
- Dictionary input is validated and passed as keyword arguments.
- A full `ToolCall` supplies its `args` and `id`.
- Pydantic field defaults are applied during validation.
- JSON-schema `args_schema` requires dictionary input.
- Runtime-injected fields are hidden from the schema sent to language models.

## Error Handling

### `handle_tool_error`

```python
False # Re-raise ToolException
True # Return the ToolException message
str # Return the configured fixed string
Callable # Return content produced by the callable
```

### `handle_validation_error`

```python
False # Re-raise the Pydantic ValidationError
True # Return "Tool input validation error"
str # Return the configured fixed string
Callable # Return the callable's string result
```

When a handled tool call has a `tool_call_id`, the handled result is returned as a `ToolMessage` with status `"error"`.

## Response Formats

### `"content"`

The `_run()` or `_arun()` result is treated as the tool-message content.

### `"content_and_artifact"`

The implementation must return:

```python
(content, artifact) # Tool-message content and raw artifact
```

A `ValueError` is raised when the result is not a two-item tuple.

---

# `InjectedToolArg: object`

Marker annotation for a tool parameter injected at runtime.

Injected parameters are excluded from the tool-call schema sent to language models.

Use it as metadata inside `Annotated`.

```python
Annotated[SomeType, InjectedToolArg] # Mark a parameter as runtime-injected
```

---

# `InjectedToolCallId: InjectedToolArg`

Marker annotation that injects the current tool-call identifier into a tool parameter.

```python
Annotated[str, InjectedToolCallId] # Receive the current tool-call identifier
```

A tool containing this annotation must be invoked with a complete `ToolCall` containing an `id`.

---

# `get_all_basemodel_annotations`

Returns field annotations from a Pydantic model and its inherited generic parents.

```python
get_all_basemodel_annotations(
    cls: TypeBaseModel | Any, # Pydantic model class or parameterized model alias
    *,
    default_to_bound: bool = True, # Whether unresolved TypeVars use their bounds or Any
) -> dict[str, type | TypeVar] # Return field names mapped to resolved annotations
```

## Behaviour

- Supports Pydantic v1 and v2 models.
- Resolves inherited generic type variables.
- Respects Pydantic field aliases.
- Excludes hidden constructor arguments added by Pydantic configuration.
- Uses a TypeVar bound when `default_to_bound=True`.
- Uses `Any` when an unresolved TypeVar has no bound.

---

# `BaseToolkit: BaseModel, ABC`

`BaseToolkit` is the abstract base class for collections of related LangChain tools.

## Abstract Method

### `get_tools`

Returns all tools contained in the toolkit.

```python
get_tools(
    self, # Current toolkit instance
) -> list[BaseTool] # Return the toolkit's tools
```

A concrete toolkit must implement this method.

## Developer-Facing Top-Level Statements

```python
FILTERED_ARGS # Function parameters excluded from generated schemas
TOOL_MESSAGE_BLOCK_TYPES # Supported structured tool-message block types
SchemaAnnotationError # Invalid BaseTool args_schema annotation error
create_schema_from_function # Function-signature to Pydantic-schema converter
ToolException # Expected and optionally handled tool execution error
ArgsSchema # Tool argument-schema type alias
MessageContentBlock # Tool-message content-block type alias
ToolExceptionHandlerOutput # Tool-error handler output type alias
BaseTool # Abstract base class for LangChain tools
InjectedToolArg # Runtime-injected tool argument marker
InjectedToolCallId # Runtime-injected tool-call ID marker
get_all_basemodel_annotations # Pydantic annotation resolver
BaseToolkit # Abstract base class for tool collections
```

In [ ]:
from pydantic import BaseModel, Field # Import Pydantic classes for the tool input schema

from langchain_core.tools import BaseTool, BaseToolkit, ToolException # Import the tool base classes and handled exception


class DivideInput(BaseModel): # Define the input schema accepted by the tool
    numerator: float = Field(description="Number that will be divided") # Define the numerator argument
    denominator: float = Field(description="Number used as the divisor") # Define the denominator argument


class DivideTool(BaseTool): # Create a concrete implementation of the abstract BaseTool class
    name: str = "divide_numbers" # Define the unique tool name
    description: str = "Divide one number by another number" # Explain when the tool should be used
    args_schema: type[BaseModel] = DivideInput # Attach the Pydantic input schema
    handle_tool_error: bool = True # Return handled ToolException messages instead of raising them

    def _run( # Implement the required synchronous tool operation
        self, # Current DivideTool instance
        numerator: float, # Validated numerator value
        denominator: float, # Validated denominator value
    ) -> float: # Return the division result
        if denominator == 0: # Check whether division by zero was requested
            raise ToolException("The denominator cannot be zero") # Raise an expected tool error

        return numerator / denominator # Calculate and return the division result


class MathToolkit(BaseToolkit): # Create a concrete implementation of the abstract BaseToolkit class
    tools: list[BaseTool] # Store the tools belonging to this toolkit

    def get_tools(self) -> list[BaseTool]: # Implement the required toolkit method
        return self.tools # Return all tools stored in the toolkit


divide_tool: DivideTool = DivideTool() # Create the concrete tool

math_toolkit: MathToolkit = MathToolkit( # Create the concrete toolkit
    tools=[divide_tool], # Add the division tool to the toolkit
) # Finish creating the toolkit

available_tools: list[BaseTool] = math_toolkit.get_tools() # Retrieve all tools from the toolkit

selected_tool: BaseTool = available_tools[0] # Select the first available tool

success_result: float = selected_tool.invoke( # Execute the tool with valid input
    {
        "numerator": 20, # Supply the numerator
        "denominator": 4, # Supply the denominator
    }
) # Finish the successful invocation

error_result: str = selected_tool.invoke( # Execute the tool with invalid division input
    {
        "numerator": 20, # Supply the numerator
        "denominator": 0, # Supply zero as the denominator
    }
) # Finish the handled-error invocation

print("Tool name:", selected_tool.name) # Display the tool name

print("Tool arguments:", selected_tool.args) # Display the generated argument schema

print("Successful result:", success_result) # Display the successful division result

print("Handled error:", error_result) # Display the handled ToolException message